# Native CLM v0 — M3L Query-Sketch Gate Diagnostic

Checkpoint-only mechanism diagnostic. This notebook does **not** retrain M3R, update Native CLM parameters, grow Cells, or consume new formal continual-learning seeds. It reconstructs the exact M3R data snapshot, downloads the exact published lineage checkpoints, tests whether compact historical query sketches can recover lineage-local affine gates without old-sample replay, and publishes lightweight evidence.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

BRANCH = 'codex/native-clm-v0-m3l-learned-lineage-gate'
REPO_URL = 'https://github.com/ArcheLabs/mini-cells.git'
ROOT = Path('/kaggle/working/mini-cells')
DATA = Path('/kaggle/working/native-clm-m3r-address-data')
CHECKPOINTS = Path('/kaggle/working/native-clm-m3r-address-checkpoints')
OUT = ROOT / 'artifacts/experiments/native-clm-v0-m3l-query-sketch-gate'

def run(cmd, cwd=None):
    print('+', ' '.join(str(x) for x in cmd), flush=True)
    subprocess.run([str(x) for x in cmd], cwd=cwd, check=True)

if not ROOT.exists():
    run(['git', 'clone', REPO_URL, str(ROOT)])
run(['git', 'fetch', '--no-tags', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}'], cwd=ROOT)
run(['git', 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], cwd=ROOT)
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[lm]'], cwd=ROOT)


In [ ]:
from kaggle_secrets import UserSecretsClient
import torch

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['GITHUB_TOKEN'] = secrets.get_secret('GITHUB_TOKEN')
assert os.environ['HF_TOKEN'], 'HF_TOKEN is missing'
assert os.environ['GITHUB_TOKEN'], 'GITHUB_TOKEN is missing'
assert torch.cuda.is_available(), 'CUDA is required for the canonical diagnostic'
assert torch.cuda.device_count() >= 2, f'expected >=2 GPUs, got {torch.cuda.device_count()}'
print('GPUs:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Boundary: checkpoint-only; no Native CLM training; M3R seeds 73611/73612/73613 are consumed inputs.')


In [ ]:
# Reconstruct and byte/SHA-verify the exact M3R A/B/C/D snapshot.
run([sys.executable, 'scripts/research/prepare_native_clm_v0_m3r_address_data.py', '--output-dir', DATA], cwd=ROOT)
manifest = json.loads((DATA / 'manifest.json').read_text())
assert manifest['exact_parent_snapshot_verified'] is True
assert manifest['parent_manifest_sha256'] == '213ddb9d093ea44fd0524e6ba6318f86a61c54270bd5cad6ddeb3233470565b0'
print('Exact M3R data snapshot: PASS')


In [ ]:
# Download only the three already-published M3R lineage checkpoints and verify their SHA-256 identities.
run([sys.executable, 'scripts/research/fetch_native_clm_v0_m3r_address_checkpoints.py', '--output-dir', CHECKPOINTS], cwd=ROOT)
ckpt_manifest = json.loads((CHECKPOINTS / 'manifest.json').read_text())
assert ckpt_manifest['revision'] == 'a23b521e137a7e44616809895d44d87cc7d6f87f'
assert sorted(r['seed'] for r in ckpt_manifest['records']) == [73611, 73612, 73613]
print('Published M3R lineage checkpoints: PASS')


In [ ]:
# Run the registered M3L mechanism diagnostic. GPU0/GPU1 process two seeds concurrently; the first free GPU receives the third.
run([
    sys.executable,
    'scripts/research/run_native_clm_v0_m3l_query_sketch_gate.py',
    '--data-dir', DATA,
    '--checkpoint-dir', CHECKPOINTS,
    '--output-dir', OUT,
    '--devices', 'cuda:0,cuda:1',
], cwd=ROOT)
result = json.loads((OUT / 'diagnostic-result.json').read_text())
print(json.dumps({
    'classification': result['classification'],
    'valid_edges': f"{result['valid_edge_count']}/{result['edge_count']}",
    'cosine_median_auc': result['current_cosine']['median'],
    'oracle_median_auc': result['offline_oracle']['median'],
    'sketch_gate_median_auc': result['sketch_gate']['median'],
    'median_oracle_recovery': result['sketch_gate']['median_normalized_oracle_excess_recovery'],
    'median_old_fpr': result['sketch_gate']['median_old_fpr'],
    'median_current_tpr': result['sketch_gate']['median_current_tpr'],
}, indent=2))


In [ ]:
# Publish JSON/CSV/MD only. No checkpoint is created by M3L.
run([
    sys.executable,
    'scripts/research/publish_native_clm_v0_m3l_query_sketch_gate.py',
    '--branch', BRANCH,
    '--output-dir', OUT,
], cwd=ROOT)
print('Published M3L classification:', result['classification'])


## Interpretation boundary

A positive `QUERY_SKETCH_GATE_FEASIBLE` result licenses a **new** online continual-language experiment that maintains query sketches during learning and transfers lineage ownership only after learning a sketch-derived gate. It does not convert M3R into a positive result. A negative result means the compact registered sketch is insufficient and should be improved before another expensive continual-language formal run.